In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
base_path = f'/Volumes/fmcg/raw/test/full_load/{data_source}/*.csv'
print(base_path)

### Bronze Processing

In [0]:
df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(base_path)\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

- Drop duplicates

In [0]:
df_duplicates = df_bronze.groupBy('product_id').count().where("count > 1")
df_duplicates.show()

In [0]:
print('Rows before duplicated dropped :', df_bronze.count())
df_silver = df_bronze.drop_duplicates(['product_id'])
print('Rows after duplicated dropped: ', df_silver.count())

In [0]:
df_silver.select('category').distinct().show()

- Title case fix

In [0]:
df_silver = df_silver.withColumn("category", F.when(F.col("category").isNull(), None).otherwise(F.initcap("category")))

In [0]:
df_silver.select('category', 'product_name').distinct().show()

- Spelling mistake

In [0]:

df_silver = (
    df_silver
    .withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("product_name"), "Protien", "Protein")
    )
)

In [0]:
display(df_silver.limit(5))

#### Standardizing customer attributes

In [0]:
# New division column
df_silver = (
  df_silver
  .withColumn(
    "division",
    F.when(F.col('category') == "Energy Bars", "Nutrition Bars")
    .when(F.col('category') == "Protein Bars", "Nutrition Bars")
    .when(F.col('category') == "Granola & Cereals", "Breakfast Foods")
    .when(F.col('category') == "Recovery Dairy", "Dairy & Recovery")
    .when(F.col('category') == "Healthy Snacks", "Healthy Snacks")
    .when(F.col('category') == "Electrolyte Mix", "Hydration & Electrolytes")
    .otherwise("Other")
  )
)

# New variant column
df_silver = df_silver.withColumn(
      "variant",
      F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

# New Product_code column
df_silver = (
  # Invalid product_id are replaced with fallback value to avoid losing fact records and ensure downstream joins remain consistent
  df_silver
  .withColumn(
    "product_code",
    F.sha2(F.col("product_name").cast("string"), 256)
  )
  # Clean product_id, keep only num IDs, else 999999
  .withColumn(
    "product_id",
    F.when(
      F.col("product_id").cast("string").rlike("^[0-9+$]"),
      F.col("product_id").cast("string")
    ).otherwise(F.lit(999999).cast('string'))
  )
  # Rename product_name -> product
  .withColumnRenamed("product_name", "product")
)

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

In [0]:
display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

### Merging data with source parent

In [0]:
delta_table = DeltaTable.forName(spark, f"fmcg.gold.dim_{data_source}")
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg.gold.dim_{data_source}")
df_child_products.show(5)

In [0]:
delta_table.alias("target").merge(
    source = df_child_products.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "division" : "source.division",
        "category" : "source.category",
        "product" : "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values = {
        "product_code" : "source.product_code",
        "division": "source.division",
        "category" : "source.category",
        "product" : "source.product",
        "variant" : "source.variant"
    }
)